# Register-number reader: train on train / val / test split
Run top to bottom. Needs: **Accelerator = GPU (T4 or P100)** and **Internet = On** (right sidebar -> Session options), and the dataset `dl-project` added as input with `train_reg.py` and `train_trocr.py` in its latest version.
Read **[VAL]** to compare settings. **[TEST]** is scored once at the end, do not tune against it.

## 1. Setup (re-run after every session restart)

In [ ]:
import glob, os, shutil, torch
src = "/kaggle/input/datasets/sanjanaps23bps1135/dl-project"
!cp -r {src}/colab_reg/* /kaggle/working/
%cd /kaggle/working
for f in ("train_reg.py", "train_trocr.py"):                      # newest scripts sit at the top of the dataset
    hit = glob.glob(f"{src}/**/{f}", recursive=True)
    top = [h for h in hit if "/colab_reg/" not in h] or hit
    shutil.copy(top[0], f"scripts/{f}"); print("using", top[0])
assert "split_of" in open("scripts/train_reg.py").read(), "old train_reg.py: upload the new version to the dataset"
print("GPU:", torch.cuda.is_available())
!pip -q install -U transformers
!ls scripts data

## 2. ResNet + BiLSTM + CTC reader (about 20 min)

In [ ]:
!python -u scripts/train_reg.py --npz --syn-epochs 100 --ft-epochs 80 2>&1 | tee resnet_log.txt

In [ ]:
# download link for the ResNet weights (14 MB)
import shutil
from IPython.display import FileLink
shutil.copy("models/reg.pt", "reg_resnet_split.pt")
FileLink("reg_resnet_split.pt")

## 3. TrOCR fine-tune (about 35 min). `--save` keeps the weights (about 670 MB)

In [ ]:
!python -u scripts/train_trocr.py --npz --epochs 6 --syn 6000 --save 2>&1 | tee trocr_log.txt

In [ ]:
# zip the TrOCR weights and make a download link
!cd models && zip -r -q ../trocr_reg_split.zip trocr_reg
from IPython.display import FileLink
FileLink("trocr_reg_split.zip")

## 4. Results to paste back
Copy the `[VAL]`, `[TEST]` and `val errors` lines from both runs. `resnet_log.txt` and `trocr_log.txt` are also in the Output tab.
Download links appear above. If a link does not work, open the **Output** tab -> `/kaggle/working/` -> hover the file -> **Download**.